# Lesson 5 — Function calling: turning a sentence into a typed call

Lesson 4 covered combining several `Score`s into one ranking. This lesson covers a different composition: mapping a natural-language request onto a call to an ordinary function, with every closed-set argument filled in and scored for confidence.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/function_calling (click to expand)</summary>

# Function calling

> Turns natural-language trading requests into calls to ordinary typed functions by mapping function names and closed-set arguments to confidence-aware TypeSafe questions.

When you order a "large iced oat latte, no sweetener," the barista does not write your
sentence down. They mark four options on a cup. This cookbook does the same thing for
a trading API: a sentence goes in, and out comes a function name and its arguments as
evaluated enums, each with a confidence.

```text
"plot rolling correlation between nvda and spy for the past month"
    rolling_correlation(symbol='NVDA', benchmark='SPY', window='1mo')   confidence 0.91

"compare nvda amd and msft over the past three months"
    compare_returns(symbols=['NVDA', 'AMD', 'MSFT'], window='3mo')      confidence 0.94

"show me apple daily with volume"
    plot_price(symbol='AAPL', resolution='1d', include_volume=True)     confidence 0.75

"what tickers do you have"
    list_symbols()                                                     confidence 1.00
```

Those calls go to ten ordinary functions in a trading assistant. Their arguments take
values from fixed lists, so they are `Literal`s already. An argument whose values come
from a fixed list is a **closed set**. When it takes one value out of that list, it gets
a `Choice` question over exactly those values, so whatever reaches the function is a
value the function accepts. You leave the functions alone. What you add is a spec that
says in plain words what each argument means.

## Closed-set shapes

The type hints already say which arguments come from a fixed list, and what is in each
list. Arguments sort into three shapes: a **choice** (one value out of a list), a **set**
(any number of values from a list), or a **flag** (a plain on/off boolean). Across the
cookbook's ten example functions, 28 arguments turn out to be fillable closed sets this
way. Free text, numbers and dates work differently: they never get a question, and the
function's own default stands if the sentence doesn't specify one.

## Write the spec

A `Literal` gives you the strings `"1mo"` and `"3mo"`. It does not say that a user typing
"this quarter" means the second one. The spec says that: one question per argument, one
line per option, a description per function, and one more question that picks between
the functions.

```json
{
  "style": {
    "question": "Does the user want a plain line or candles?",
    "stated": "Does the user say how the chart should be drawn, such as a line, candles, or OHLC bars?",
    "options": {
      "line": "a simple line through the closing prices",
      "candles": "a candlestick or OHLC chart, showing each bar's open, high, low and close"
    }
  }
}
```

The option keys are the strings the function takes, so nothing has to map a label back to
an argument afterwards. `stated` makes an argument optional: it is a second yes/no
question asking whether the command says anything about that argument at all. When the
answer is no, the call leaves that argument out and the function's own default applies.

A set argument gets its question once per member, with `{}` standing in for the member
name — `"Does the user want {} in the comparison?"` becomes one question per ticker.

Write each question about the idea rather than the words a user might pick, because the
match is on meaning: "is amd tracking nvidia lately" reaches `rolling_correlation` even
though neither *tracking* nor *lately* appears anywhere in the spec. Avoid naming a
question after its parameter — "Which resolution?" gives the command nothing to match
against.

## Read the confidence

`confidence` reports the least certain judgement in the call, rather than the product of
all of them, since one wrong argument is enough to spoil the result. A product answers a
different question ("is every part right"), and it falls as a function takes more
arguments, whether or not any one judgement is shaky.

Where that number came from, argument by argument, for `"is amd tracking nvidia lately"`
→ `rolling_correlation(symbol='AMD', benchmark='NVDA')` at confidence `0.82`:

| Argument | Value | Probability | Top distribution |
| --- | --- | --- | --- |
| `symbol` | `'AMD'` | 0.87 | AMD 0.87, NVDA 0.13, AAPL 0.00 |
| `benchmark` | `'NVDA'` | 0.78 | NVDA 0.92, AMD 0.08, AAPL 0.00 |
| `window` | omitted, default stands | 0.96 | — |
| `resolution` | omitted, default stands | 0.99 | — |

Weakest argument: `benchmark`, at 0.78 — that's why the call's overall confidence is 0.82
and not higher. `window` and `resolution` are both omitted here, because "lately" does not
say how far back or on what bars, so the function runs on its own defaults. That is what
the `stated` question is for: without it, the choice would have to name some window, and
it would have named one confidently — just not necessarily the right one.

> This cookbook's full version wires up ten real trading functions, a `Dispatcher` class,
> and fourteen worked commands with charted output — trimmed here to the mapping technique
> itself, since the trading infrastructure isn't something we can run in this repo. Read
> the full cookbook at [docs.typesafe.ai/cookbooks/function_calling](https://docs.typesafe.ai/cookbooks/function_calling) any time.

</details>

## The core idea

Function calling here isn't the model writing code or picking a function by name out of thin air — it's the same three primitives you already know, aimed at a specific shape of problem: **which function**, and **which value, from a fixed list, for each of its arguments**.

- Picking the function is a `Choice` over the function names/descriptions.
- Each closed-set argument is its own `Choice` (or a `Noul` per member, for a set-valued argument).
- An *optional* argument gets a companion `Noul` asking "did the user even say anything about this?" — if not, skip the argument and let the function's own default apply, rather than forcing a guess.
- The call's overall confidence is the **minimum** across every question actually used — not a product, and not an average. One weak argument is enough to make the whole call unreliable, no matter how confident the rest were.

Since every one of those questions can go in a single request, this is speculative fan-out (lesson 3) aimed specifically at "which function, with what arguments".

In [1]:
import { client } from "../assets/typesafe-client.ts";
import { choice, noul } from "npm:@typesafe-ai/sdk";

const command = "text me when you're done, this is pretty urgent";

const response = await client.systemOne({
  state: command,
  questions: {
    tool: choice("Which action is the user asking for?", {
      scheduleReminder: "Set a reminder or notify someone at a later point",
      summarizeSentiment: "Judge the emotional tone of a piece of text",
    }),
    urgency: choice("How urgent does the user want the reminder to be?", {
      low: "No particular rush",
      medium: "Should happen reasonably soon",
      high: "Needs immediate attention",
    }),
    channel: choice("Which channel does the user want the reminder sent through?", {
      email: "Email",
      sms: "Text message / SMS",
      push: "A push notification",
    }),
    channelStated: noul(
      "The user says which channel to use (email, text, or push notification)",
    ),
    tone: choice("What is the emotional tone of the text?", {
      positive: "Positive",
      negative: "Negative",
      neutral: "Neutral",
    }),
  },
});

const { tool, urgency, channel, channelStated } = response.answers;

console.log("tool:", tool.choice, "confidence", tool.confidence.toFixed(2));

if (tool.choice === "scheduleReminder") {
  const args: Record<string, unknown> = { urgency: urgency.choice };
  let callConfidence = Math.min(tool.confidence, urgency.confidence);
  if (channelStated.noul > 0.5) {
    args.channel = channel.choice;
    callConfidence = Math.min(callConfidence, channel.confidence);
  }
  console.log("scheduleReminder(", args, ") confidence", callConfidence.toFixed(2));
} else {
  console.log("summarizeSentiment(", { tone: response.answers.tone.choice }, ")");
}

tool: scheduleReminder confidence 1.00
scheduleReminder( { urgency: "high", channel: "sms" } ) confidence 0.97
scheduleReminder( { urgency: "high", channel: "sms" } ) confidence 0.97


## What just happened

Five questions, one call, two candidate functions. `tool` should land on `scheduleReminder` — nothing in "text me when you're done, this is pretty urgent" is about judging tone. `channelStated` should come back high (the command says "text me" — that's a channel), so `channel` gets included in the final call; `tone`'s answer exists in the response but never gets read, because the chosen tool doesn't need it. That's the fan-out payoff again: asking it cost almost nothing, and ignoring it costs nothing either.

**Try it yourself:** rewrite the command to drop any mention of a channel entirely (just "remind me later, it's urgent") and predict whether `channelStated` flips low enough that `channel` gets omitted from the final call. Then try a command that's genuinely about judging tone, and predict which tool wins.

## Next up

That closes out the use-case patterns this series set out to cover — fan-out, composite scoring, function calling. Next lesson: a dedicated look at confidence and escalation thresholds, and where the vendor's own marketing claims part ways with what the technical docs actually promise.

Questions? Ask your agent — it has the same research this lesson was built from, in `research/typesafe-ai.md`.